# FaunaVision 02: fine-tuned models, distilled student and ONNX export

Runs on a free Kaggle GPU notebook (GPU T4, Internet on). Launch from VS Code with `kaggle kernels push -p kaggle/02_models`.

Models 2 to 6 do not fit in one 12-hour session, so this notebook runs **one stage per version**:

| `STAGE` | Trains | Needs from earlier stages |
|---|---|---|
| `resnet50` | Model 2: ResNet-50 baseline | nothing |
| `dinov2_ft` | Main model: DINOv2 ViT-B/14 at 224 px | nothing |
| `dinov2_448` | Model 5: main model refined at 448 px | `dinov2_ft` |
| `student` | Model 6: EfficientNet-B0 distilled from model 5 | `dinov2_448` |
| `export` | `model.onnx` + `model.json` for the API (step 8) | `student` |

**Before the first run**, attach the crops dataset you saved from notebook 01 (*Add Input*) and set `CROPS_DIR` below.

**Between stages**, attach this notebook's own previous output as an input and set `PREV_OUTPUTS` to it, so later
stages find earlier checkpoints. If a session drops mid-stage, re-run the same stage with the previous output attached:
training resumes from the last epoch.

In [ ]:
STAGE = "resnet50"  # resnet50 | dinov2_ft | dinov2_448 | student | export
CROPS_DIR = "/kaggle/input/faunavision-crops/crops"  # crops dataset saved from notebook 01
PREV_OUTPUTS = []  # e.g. ["/kaggle/input/faunavision-02-models"]: earlier versions of this notebook
REVIEW_RECALL = 0.8  # export only: share of validation lookalike errors the review flag must catch

REPO_URL = "https://github.com/<your-username>/faunavision.git"  # set this

In [ ]:
!git clone -q $REPO_URL /kaggle/working/faunavision
%cd /kaggle/working/faunavision
!pip install -q -e ".[export]" ijson timm

## 1. Manifest (same deterministic splits as notebook 01)

In [ ]:
!mkdir -p data/raw/lila
!wget -q -O data/raw/lila/ss2024_meta.zip https://storage.googleapis.com/public-datasets-lila/snapshot-safari-2024-expansion/snapshot_safari_2024_metadata.zip
!wget -q -O data/raw/lila/md_ss2024_rde.json.zip https://lila.science/public/lila-md-results/snapshot_safari_2024_lila_mdv5a-rde_0.100_0.850_15_0.200.json.zip
!python scripts/build_camtrap_index.py
!python scripts/make_manifest.py --test-projects PLN,KRU,CDB

## 2. Restore checkpoints and results from earlier stages

In [ ]:
import shutil
from pathlib import Path

OUT = Path("/kaggle/working/out")  # everything worth keeping ends up here
for prev in map(Path, PREV_OUTPUTS):
    for sub in ("checkpoints", "results"):
        src = next((p for p in [prev / sub, prev / "out" / sub] if p.exists()), None)
        if src is None:
            continue
        dst = Path("checkpoints") if sub == "checkpoints" else Path("reports/results")
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"restored {src} -> {dst}")
!ls checkpoints reports/results 2>/dev/null

## 3. Run the stage

In [ ]:
DATA = f"--crops {CROPS_DIR} --resume"
TEACHER = "checkpoints/dinov2_b14_448/best.pt"

COMMANDS = {
    "resnet50": f"python scripts/train.py --model resnet50.a1_in1k --run resnet50 --epochs 15 {DATA}",
    "dinov2_ft": f"python scripts/train.py --model vit_base_patch14_dinov2.lvd142m --run dinov2_b14_ft "
                 f"--lr 5e-4 --layer-decay 0.7 --batch-size 48 {DATA}",
    "dinov2_448": f"python scripts/train.py --model vit_base_patch14_dinov2.lvd142m --run dinov2_b14_448 "
                  f"--img-size 448 --epochs 3 --lr 1e-4 --layer-decay 0.7 --warmup-epochs 0.2 --batch-size 16 "
                  f"--init checkpoints/dinov2_b14_ft/best.pt {DATA}",
    "student": f"python scripts/train.py --model efficientnet_b0.ra_in1k --run student_b0 "
               f"--teacher {TEACHER} --kd-alpha 0.7 --kd-temp 2 --epochs 20 {DATA}",
    "export": "python scripts/export_onnx.py --ckpt checkpoints/student_b0/best.pt --out exports/model.onnx "
              f"--metrics reports/results/student_b0/metrics.json --review-recall {REVIEW_RECALL}",
}
NEEDS = {"dinov2_448": "checkpoints/dinov2_b14_ft/best.pt", "student": TEACHER,
         "export": "checkpoints/student_b0/best.pt"}
if STAGE not in COMMANDS:
    raise ValueError(f"unknown STAGE {STAGE!r}")
if STAGE in NEEDS and not Path(NEEDS[STAGE]).exists():
    raise FileNotFoundError(f"{STAGE} needs {NEEDS[STAGE]}: attach the earlier output and set PREV_OUTPUTS")

# --resume needs last.pt next to best.pt; outputs are copied back below either way
print(COMMANDS[STAGE])
!{COMMANDS[STAGE]}

## 4. Keep the outputs

Checkpoints, results and (for `export`) the ONNX model are copied to `/kaggle/working/out`.

In [ ]:
RUNS = {"resnet50": "resnet50", "dinov2_ft": "dinov2_b14_ft", "dinov2_448": "dinov2_b14_448", "student": "student_b0"}
OUT.mkdir(exist_ok=True)
shutil.copytree("checkpoints", OUT / "checkpoints", dirs_exist_ok=True)
shutil.copytree("reports/results", OUT / "results", dirs_exist_ok=True)
if STAGE == "export":
    shutil.copytree("exports", OUT / "exports", dirs_exist_ok=True)
!du -sh /kaggle/working/out/*

In [ ]:
import json
from IPython.display import Markdown

if STAGE == "export":
    meta = json.load(open("exports/model.json"))
    print(json.dumps({k: meta[k] for k in ("model_name", "img_size", "temperature", "review_margin")}, indent=1))
    print(json.dumps(meta.get("review_tuning"), indent=1))
else:
    r = json.load(open(f"reports/results/{RUNS[STAGE]}/metrics.json"))
    NL = chr(10)
    lines = ["| split | top-1 | top-5 | macro-F1 | ECE |", "|---|---|---|---|---|"]
    lines += [f"| {s} | {r[s]['top1']:.3f} | {r[s]['top5']:.3f} | {r[s]['macro_f1']:.3f} | {r[s]['ece']:.3f} |"
              for s in ("val", "test")]
    display(Markdown(NL.join(lines)))